# 13 — Train Video Model

Fine-tunes every model in `config.VIDEO_MODELS_TO_RUN` on the video clips
produced by `04_preprocess_video.ipynb`.

**Depends on:** `04_preprocess_video.ipynb` having been run (frame cache must exist).

## Anti-overfitting measures (all on by default)
| Measure | Config key |
|---|---|
| Backbone freeze warm-up | `VIDEO_FREEZE_EPOCHS` |
| Differential learning rates | `VIDEO_BACKBONE_LR_MULT` |
| Label smoothing | `VIDEO_LABEL_SMOOTHING` |
| Class-balanced sampler | `VIDEO_BALANCED_SAMPLER` |
| Temporal-consistent augmentation + RandomErasing | `VIDEO_RANDOM_ERASING_P` |
| Gradient accumulation | `VIDEO_GRAD_ACCUM_STEPS` |

## Outputs (per model)
```
../experiments/video_unimodal/<version>/<slug>/
    model/best_model.pt   ← best checkpoint (by dev macro-F1)
    model/arch.txt        ← architecture name
    history.csv
    config.json
    train_metrics.json
```

In [1]:
import os
import sys

_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'src')) else os.path.abspath(os.path.join(_here, '..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

In [2]:
from src.config import (
    DEVICE,
    VERSION,
    VIDEO_MODELS_TO_RUN,
    VIDEO_CACHE_DIR,
    VIDEO_TRAIN_BATCH_SIZE,
    VIDEO_GRAD_ACCUM_STEPS,
    VIDEO_TRAIN_EPOCHS,
    VIDEO_TRAIN_LR,
    VIDEO_WEIGHT_DECAY,
    VIDEO_WARMUP_RATIO,
    VIDEO_GRAD_CLIP_NORM,
    VIDEO_LABEL_SMOOTHING,
    VIDEO_BACKBONE_LR_MULT,
    VIDEO_FREEZE_EPOCHS,
    VIDEO_HEAD_DROPOUT,
    VIDEO_BALANCED_SAMPLER,
    VIDEO_RANDOM_ERASING_P,
    VIDEO_IMG_SIZE,
    VIDEO_T_CACHE,
)
from src.data.video_preprocessing import load_video_metadata, _frame_cache_dir
from src.data.video_dataset import LABEL2ID, ID2LABEL
from src.training.train_video import train_one_video

print('Device          :', DEVICE)
print('Models to train :', VIDEO_MODELS_TO_RUN)

Device          : cuda
Models to train : ['mvit_v2_s', 's3d']


## Load metadata

In [3]:
from src.config import FACE_MELD_CLASSES

_label2id = {cls.lower(): i for i, cls in enumerate(FACE_MELD_CLASSES)}

def _load_split(split):
    meta = load_video_metadata(split, version=VERSION)
    meta = meta[meta['video_exists'] & meta['emotion_norm'].isin(_label2id)].reset_index(drop=True)
    meta['label_id'] = meta['emotion_norm'].map(_label2id).astype(int)
    return meta

train_df = _load_split('train')
dev_df   = _load_split('dev')
test_df  = _load_split('test')

print(f'Train utterances : {len(train_df)}')
print(f'Dev   utterances : {len(dev_df)}')
print(f'Test  utterances : {len(test_df)}')
print(f'Classes          : {list(LABEL2ID.keys())}')

Train utterances : 9989
Dev   utterances : 1108
Test  utterances : 2610
Classes          : ['anger', 'disgust', 'fear', 'happiness', 'neutral', 'sadness', 'surprise']


## Locate frame-cache directories

In [4]:
cache_dirs = {
    split: _frame_cache_dir(split, VERSION, VIDEO_CACHE_DIR)
    for split in ['train', 'dev', 'test']
}

for split, path in cache_dirs.items():
    n = len([f for f in os.listdir(path) if f.endswith('.npy')]) if os.path.isdir(path) else 0
    ok = '✓' if n > 0 else '✗  ← run 04_preprocess_video.ipynb first'
    print(f'  [{split:5s}] {n:5d} cached clips  {ok}')

  [train]  9988 cached clips  ✓
  [dev  ]  1108 cached clips  ✓
  [test ]  2610 cached clips  ✓


## Optional — override hyperparameters

In [5]:
# ── Per-model overrides (None → use config defaults) ──────────────────────────
batch_size_override   = None   # e.g. 2 for MViT on small GPU
grad_accum_override   = None

# ── Shared overrides ─────────────────────────────────────────────────────────
epochs           = VIDEO_TRAIN_EPOCHS
lr               = VIDEO_TRAIN_LR
weight_decay     = VIDEO_WEIGHT_DECAY
warmup_ratio     = VIDEO_WARMUP_RATIO
grad_clip        = VIDEO_GRAD_CLIP_NORM
label_smoothing  = VIDEO_LABEL_SMOOTHING
backbone_lr_mult = VIDEO_BACKBONE_LR_MULT
freeze_epochs    = VIDEO_FREEZE_EPOCHS
head_dropout     = VIDEO_HEAD_DROPOUT
balanced_sampler = VIDEO_BALANCED_SAMPLER
erasing_p        = VIDEO_RANDOM_ERASING_P

print('Running with:')
print(f'  epochs={epochs}, lr={lr}, wd={weight_decay}')
print(f'  freeze={freeze_epochs}, backbone_mult={backbone_lr_mult}')
print(f'  label_smoothing={label_smoothing}, balanced={balanced_sampler}')

Running with:
  epochs=20, lr=0.0001, wd=0.01
  freeze=3, backbone_mult=0.1
  label_smoothing=0.1, balanced=True


## Train

In [ ]:
run_dirs = []

for model_name in VIDEO_MODELS_TO_RUN:
    run_dir = train_one_video(
        model_name       = model_name,
        train_df         = train_df,
        dev_df           = dev_df,
        test_df          = test_df,
        cache_dirs       = cache_dirs,
        batch_size       = batch_size_override,
        grad_accum       = grad_accum_override,
        epochs           = epochs,
        lr               = lr,
        weight_decay     = weight_decay,
        warmup_ratio     = warmup_ratio,
        grad_clip        = grad_clip,
        label_smoothing  = label_smoothing,
        backbone_lr_mult = backbone_lr_mult,
        freeze_epochs    = freeze_epochs,
        head_dropout     = head_dropout,
        balanced_sampler = balanced_sampler,
        erasing_p        = erasing_p,
        version          = VERSION,
        device           = DEVICE,
    )
    run_dirs.append(run_dir)

print('\nAll run directories:')
for d in run_dirs:
    print(' ', d)


  Video model     : mvit_v2_s
  Run dir         : ../experiments/video_unimodal\v1\mvit_v2_s__run5
  Device          : cuda
  T (frames/clip) : 16
  Batch size      : 4  |  Grad accum: 4  (eff. batch=16)
  Freeze epochs   : 3
  Backbone LR     : 1.00e-05  |  Head LR: 1.00e-04
  Label smoothing : 0.1
  Balanced sampler: True


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\src\data\video_dataset.py:419: UserWarning: FrameCacheVideoDataset: skipping 1 row(s) whose .npy cache file was not found in '../Data/processed_meld_video/cache\npy\v1\train'.
  train_ds = FrameCacheVideoDataset(


  Params          : 34,235,527 trainable / 34,235,527 total
  Backbone frozen : 5,383 params active for first 3 epoch(s).


e:\Licenta\Multimodal-Emotion-Recognition---Bachelor-Thesis\.venv\Lib\site-packages\torch\optim\lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


  [Epoch 01/20] [frozen] train_loss=1.9583  dev_loss=1.9215  dev_macro_f1=0.1572
    ↳ New best dev macro-F1: 0.1572 — checkpoint saved.
  [Epoch 02/20] [frozen] train_loss=1.9434  dev_loss=1.9349  dev_macro_f1=0.1615
    ↳ New best dev macro-F1: 0.1615 — checkpoint saved.
  [Epoch 03/20] [frozen] train_loss=1.9315  dev_loss=1.9318  dev_macro_f1=0.1517

  [Epoch 4] Backbone unfrozen — 34,235,527 params now active.
  [Epoch 04/20] train_loss=1.9074  dev_loss=1.9322  dev_macro_f1=0.1725
    ↳ New best dev macro-F1: 0.1725 — checkpoint saved.
  [Epoch 05/20] train_loss=1.8831  dev_loss=1.9049  dev_macro_f1=0.1816
    ↳ New best dev macro-F1: 0.1816 — checkpoint saved.
  [Epoch 06/20] train_loss=1.8523  dev_loss=1.9106  dev_macro_f1=0.1770
  [Epoch 07/20] train_loss=1.8122  dev_loss=1.9304  dev_macro_f1=0.1824
    ↳ New best dev macro-F1: 0.1824 — checkpoint saved.
  [Epoch 08/20] train_loss=1.7835  dev_loss=1.9122  dev_macro_f1=0.1710
  [Epoch 09/20] train_loss=1.7534  dev_loss=1.8947  de

## Training history

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

valid_dirs = [d for d in run_dirs if d and os.path.exists(os.path.join(d, 'history.csv'))]

if valid_dirs:
    fig, axes = plt.subplots(1, len(valid_dirs), figsize=(8 * len(valid_dirs), 4), squeeze=False)

    for ax, run_dir in zip(axes[0], valid_dirs):
        hist = pd.read_csv(os.path.join(run_dir, 'history.csv'))
        ax.plot(hist['epoch'], hist['train_loss'],   label='train loss',   marker='o')
        ax.plot(hist['epoch'], hist['dev_loss'],     label='dev loss',     marker='s')
        ax.plot(hist['epoch'], hist['dev_macro_f1'], label='dev macro-F1', marker='^')

        # Mark freeze boundary
        if 'frozen' in hist.columns:
            thaw = hist[~hist['frozen']]['epoch'].min() if (~hist['frozen']).any() else None
            if thaw:
                ax.axvline(thaw - 0.5, color='gray', linestyle=':', alpha=0.7, label='unfreeze')

        ax.set_xlabel('Epoch')
        ax.set_title(os.path.basename(run_dir).replace('_', '-'))
        ax.legend()

    plt.suptitle('Video training history', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('No history files found yet — run training first.')

## Training summary

In [ ]:
import json

rows = []
for run_dir in run_dirs:
    if not run_dir:
        continue
    tm_path = os.path.join(run_dir, 'train_metrics.json')
    if not os.path.exists(tm_path):
        continue
    tm = json.load(open(tm_path))
    rows.append({
        'model':           tm.get('model_name'),
        'best_dev_mf1':    tm.get('best_dev_macro_f1'),
        'total_params':    tm.get('total_params'),
        'train_time_min':  round(tm.get('training_time_s', 0) / 60, 1),
        'avg_gpu_power_w': tm.get('avg_gpu_power_w'),
        'energy_kwh':      tm.get('energy_kwh'),
    })

if rows:
    pd.DataFrame(rows)
else:
    print('No train_metrics.json files found yet.')